<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/05_representation_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Looking at the learned representation (diagnostics only)

We have a trained encoder. What did it actually learn? Here we turn each video into a single feature vector with the frozen target encoder and project those vectors to two dimensions with t-SNE and UMAP, to *see* whether normal, ms, and pd land in different regions.

> **These pictures are diagnostics, not evidence.** Two honest cautions run through this notebook. First, t-SNE and UMAP distort distances; a clean-looking blob can be an artifact of the projection. Second, and more important, apparent separation can come from a **shortcut** (camera frame rate, body size, how visible the joints are) rather than from gait. So we plot the S-JEPA embedding *and* a cheap nuisance feature side by side: if the nuisance separates just as well, the pretty S-JEPA plot is not telling us about gait. The verdict lives in notebook 06's leakage-safe scores, never in a scatter plot.

We compare the label-free checkpoint from notebook 03 against the SSL-continued one from notebook 04. No test labels are ever used to fit, select, or color anything beyond the plain class of each point.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'vicreg_clusters.svg')))

## Embed every video with the frozen encoder

For each video we mean-pool the encoder's features over its windows and over a **fixed**, seeded read-out pool of target tokens (the same pool used in notebooks 04 and 06). This pool is never chosen from labels. We do it for both checkpoints.


In [ ]:
import numpy as np, torch
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.train_v2 import load_checkpoint_v2
from sjepa.masking_v2 import sample_target_mask
from sjepa.data import load_index, sliding_windows

cfg = get_config(); device = pick_device()
records = load_index(KEYPOINTS_DIR)
readout = sample_target_mask(cfg.num_joints, cfg.num_time_tokens,
                             np.random.default_rng(0), target_ratio=0.6)
tm = torch.from_numpy(readout).to(device)

def embed_all(ckpt):
    m = build_model(cfg, device=device, repaired=True)
    load_checkpoint_v2(ckpt, m, map_location=device)
    vecs, labels = [], []
    for r in records:
        w = sliding_windows(r.load_norm(), cfg.window_frames, cfg.window_stride)
        x = torch.from_numpy(w).float().to(device)
        with torch.no_grad():
            vecs.append(m.embed(x, tm).mean(0).cpu().numpy())
        labels.append(r.label)
    return np.stack(vecs), labels

E_base, y = embed_all(ARTIFACT_DIR / 'sjepa_ssl.pt')
E_cont, _ = embed_all(ARTIFACT_DIR / 'sjepa_ssl_continued.pt')
np.savez(ARTIFACT_DIR / 'embeddings_3class.npz', E_base=E_base, E_continued=E_cont,
         labels=np.array(y))
print('embedded', len(y), 'videos into', E_cont.shape[1], 'dimensions')

## A nuisance baseline to keep us honest

This is the cheapest possible 'representation': the per-joint mean and spread of the raw visibility channel, which we already know tracks the acquisition domain (the MS clips were all filmed at 60fps). If this separates the classes as cleanly as S-JEPA does, then a tidy S-JEPA scatter is not evidence of learned gait.


In [ ]:
def nuisance_vec(r):
    vis = r.load_raw()[:, :, 2]
    return np.nan_to_num(np.concatenate([np.nanmean(vis, 0), np.nanstd(vis, 0)]))
N_nuis = np.stack([nuisance_vec(r) for r in records])
print('nuisance feature shape:', N_nuis.shape)

## Project and plot

t-SNE squeezes the high-dimensional vectors into a plane while trying to keep neighbors together. We color one hue per condition, and we place the nuisance baseline in the same row for the comparison the caution above demands.


In [ ]:
from sklearn.manifold import TSNE
from sjepa.viz import scatter_2d
import matplotlib.pyplot as plt

def tsne2d(E):
    perp = min(15, max(2, len(E)//3))
    return TSNE(n_components=2, perplexity=perp, random_state=42, init='pca').fit_transform(E)

fig, ax = plt.subplots(1, 3, figsize=(15,4.4))
scatter_2d(tsne2d(E_base), y, ax[0], 't-SNE: S-JEPA (label-free, nb 03)')
scatter_2d(tsne2d(E_cont), y, ax[1], 't-SNE: S-JEPA (SSL continued, nb 04)')
scatter_2d(tsne2d(N_nuis), y, ax[2], 't-SNE: nuisance (visibility only)')
plt.tight_layout(); plt.savefig(IMAGES_DIR / 'tsne_sjepa_vs_nuisance.png', dpi=130)
plt.show()

In [ ]:
# UMAP view (falls back gracefully if umap-learn is missing).
import matplotlib.pyplot as plt
from sjepa.viz import scatter_2d
try:
    import umap
    def umap2d(E):
        nn = min(15, max(2, len(E)//3))
        return umap.UMAP(n_neighbors=nn, min_dist=0.3, random_state=42).fit_transform(E)
    fig, ax = plt.subplots(1, 3, figsize=(15,4.4))
    scatter_2d(umap2d(E_base), y, ax[0], 'UMAP: S-JEPA (nb 03)')
    scatter_2d(umap2d(E_cont), y, ax[1], 'UMAP: S-JEPA (nb 04)')
    scatter_2d(umap2d(N_nuis), y, ax[2], 'UMAP: nuisance (visibility)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('UMAP not available, skipping:', e)

## Put a (descriptive) number on the separation

The silhouette score summarizes how tight and well separated the class clusters are, from -1 to +1. We report it for all three embeddings so the S-JEPA numbers are read *against* the nuisance number, not in isolation. On ~47 videos this is noisy and purely descriptive: it is computed on the whole set, so it is **not** an out-of-sample score and must not be used to pick a model. Model selection happens only through the leakage-safe folds in notebook 06.


In [ ]:
from sjepa.eval import silhouette
s_base = silhouette(E_base, y)
s_cont = silhouette(E_cont, y)
s_nuis = silhouette(N_nuis, y)
print(f'silhouette (descriptive, whole set):')
print(f'  S-JEPA label-free (nb 03): {s_base:.3f}')
print(f'  S-JEPA SSL continued (nb 04): {s_cont:.3f}')
print(f'  nuisance (visibility only): {s_nuis:.3f}')
if s_nuis >= max(s_base, s_cont):
    print('Note: the nuisance baseline separates at least as well -- a clean S-JEPA plot',
          'here would NOT be evidence of learned gait. See notebook 06.')